# Lesson 02 — Lucas-Kanade Sparse Optical Flow

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture('sample_video.mp4')
ret, old_frame = cap.read()
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)

# Track these specific corner points
p0 = cv2.goodFeaturesToTrack(old_gray, maxCorners=100, qualityLevel=0.3, minDistance=7)

lk_params = dict(winSize=(15,15), maxLevel=2,
                 criteria=(cv2.TERM_CRITERIA_EPS|cv2.TERM_CRITERIA_COUNT,10,0.03))

colors = np.random.randint(0,255,(100,3))
mask   = np.zeros_like(old_frame)

for _ in range(20):
    ret, frame = cap.read()
    if not ret or p0 is None: break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Track: find new positions of p0 in next frame
    p1, st, _ = cv2.calcOpticalFlowPyrLK(old_gray, gray, p0, None, **lk_params)

    good_new = p1[st==1]
    good_old = p0[st==1]

    for i,(new,old) in enumerate(zip(good_new,good_old)):
        a,b = new.ravel().astype(int)
        c,d = old.ravel().astype(int)
        mask  = cv2.line(mask,(a,b),(c,d),colors[i%len(colors)].tolist(),2)
        frame = cv2.circle(frame,(a,b),4,colors[i%len(colors)].tolist(),-1)

    display = cv2.add(frame, mask)
    old_gray = gray.copy()
    p0 = good_new.reshape(-1,1,2)

cap.release()
plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(display, cv2.COLOR_BGR2RGB))
plt.title('Lucas-Kanade sparse flow — colored trails show each tracked point'); plt.axis('off'); plt.show()

## Key Takeaway
Lucas-Kanade tracks specific points across frames. Trails accumulate to show motion history.
Use `goodFeaturesToTrack` to seed the initial points.